# QELM on the measurement record M, sweep over p

Experiment `experiments/QELM_M/exp_QELM_M.py` (issue #94). For each sample, the reservoir (disordered TFIM, with an ancilla entangled with the charge of the system) evolves for `evolution.t`, split into `Ns` steps. After each step, each system qubit is measured in Z with probability `p`. The features are the measurement record `M` (0 for an unmeasured qubit), and the target `Y` is the outcome ±1 of the ancilla `R` measured in Z at the end. `R` is a reference qubit: the TFIM acts on the system only, so `Y` is the charge sector the system started in, +1 or -1 with probability 1/2 whatever `p` (issue #104).

This notebook launches the sweep with `experiments/QELM_M/exp_QELM_M.sh`, reads back the folders `outputs/QELM_M/<date_time>/N<N>_seed<seed>_p<p>/`, and plots against `p` two separate figures (issue #174): one for the **performance** of the readout, one for the **entanglement** of the final state. Each metric is averaged over the seeds, with the standard error as error bars. The last section sweeps `p` again for two transverse fields, `h = J` and `h = 10 J`, and two sizes, `N = 4` and `N = 6`.

At p = 1, the last round measures the parity of the system charge, which the TFIM conserves, so the record fixes `Y` and the accuracy reaches 1: the charge is learnable. Before issue #104 the target was `<Z_0>` on the final state, with `R` evolving too; it was ±1 at p = 1 and continuous below, so it changed with `p` itself. The label `Y` is now fixed before any measurement, and only what the record reveals of it changes with `p`.

### Performance

- **Accuracy** of the sign of the prediction on the test labels `Y`. Its chance level shares its panel: `accuracy_baseline`, the accuracy of always predicting the majority sign of the training set: a finite test set is never exactly balanced (issue #130), so a readout that learns nothing scores its share, not exactly 0.5.
- **R2** of the prediction, the continuous counterpart of the accuracy.

### Entanglement (bits)

- **Ancilla entropy** `S_vN`, the purification order parameter of the measurement-induced transition: `S = 1` bit as long as the ancilla stays entangled with the system, `S = 0` once the measurements have purified it. Its Rényi-2 entropy shares the panel, dashed (`S_2 <= S_vN`).
- **Half-system entropy** `S_half`, the von Neumann entropy of the first `Ns // 2` system qubits (issue #174): the entanglement across the middle of the chain, which the measurements destroy.
- **Mean single-qubit entropy** `S_vN_1q`, the entanglement of each system qubit with all the rest, averaged over the system qubits, with its Rényi-2 version dashed.

### Metrics left out of the figures

The runs still record them, and the table below prints them, but each repeats a curve already drawn:

- `S_renyi_1` is the `n -> 1` limit of the Rényi family, that is the von Neumann entropy: it equals `S_vN` sample by sample (issue #119).
- `S_vN_sys`, the entropy of the system sector: the global state stays pure (unitary evolution and projective measurements), so it equals `S_vN` at every sample, and `S_renyi_2_sys` is the Rényi-2 entropy of the ancilla, the dashed curve of the ancilla panel (issue #124).
- `nmse = MSE / mean(Y^2) = MSE` for labels ±1, while `R2 = 1 - MSE / var(Y)` with `var(Y) = 1 - mean(Y)^2`, close to 1: `R2 ≈ 1 - nmse`, the same information.
- `balanced_accuracy` averages, over the two signs, the fraction recovered. With labels of probability 1/2 it follows the accuracy, and the gap to the chance level, drawn on the accuracy panel, already separates learning from the imbalance of the test set (issue #130).
- The panels of the standard error alone: the error bars carry it.

No Bayes ceiling is drawn (issue #195). The former `accuracy_optimal`, `sign(<Z_0>)` on the exact final state of `R`, used `P(Y | rho_R)`, which knows the Hamiltonian of each sample, while the readout only sees the record `M`: a ceiling for it would need `P(Y | M)`. Comparing the two misread the performance figure, so the metric and `<Z_0>` were removed from `exp_QELM_M.py`.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_qelm_m_notebook.py` does for a small sweep). Notation, the same in every notebook: `N` is the total number of qubits, ancilla included, and `L` the number of samples per run (`samples.L` in the config). Each sweep goes to this notebook's own folder, `outputs/QELM_M/<date_time>/`, and the notebook reads back exactly that folder (issue #151). `RUN = False` reads back the latest sweep of this folder without launching a new one. The parameters of the last section, the sweep over `h` and `N`, are set in that section.

In [ ]:
L = os.environ.get("L", "500")  # samples per run (samples.L)
N_VALUES = os.environ.get("N_VALUES", "4")  # total qubits N, ancilla included
SEEDS = os.environ.get("SEEDS", "42 43 44")
P_VALUES = os.environ.get("P_VALUES", "0.0 0.1 0.2 0.3 0.5 0.8 1.0")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151): no other notebook writes
# there, so the cells below never read another notebook's sweep.
EXPERIMENT = "QELM_M"

if RUN:
    # Create the sweep folder here and hand it to the launcher (OUT_DIR):
    # the cells below read this sweep and no other.
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "L": L,
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        # The field of the config, h = 10 J, even if H_VALUES is set for
        # the last section (issue #174).
        "H_VALUES": "",
        "OUT_DIR": str(sweep_dir),
    }
    n_runs = len(N_VALUES.split()) * len(SEEDS.split()) * len(P_VALUES.split())
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/QELM_M/exp_QELM_M.sh"], total=n_runs, env=env
    )
else:
    # RUN = False: the latest sweep of this notebook only.
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the metrics

In [ ]:
from QELM.utils import load_sweep, summarize

# (key in metrics.json, label): every metric of the runs, printed in the
# table. A metric missing from an older sweep is left out: the ancilla
# entropies came with issue #119, the accuracy and the system entropies with
# issue #124, the accuracy baseline and the balanced accuracy with issue
# #130, S_half with #174. The Bayes accuracy of #104 was removed (#195).
METRICS = [
    ("accuracy", "accuracy of label Y (test)"),
    ("accuracy_baseline", "chance level of accuracy (majority sign)"),
    ("balanced_accuracy", "balanced accuracy of label Y (test)"),
    ("r2", "R2 (test)"),
    ("nmse", "NMSE (test)"),
    ("S_vN_mean", "S_vN ancilla (bits)"),
    ("S_renyi_1_mean", "S_Rényi-1 ancilla (bits)"),
    ("S_vN_sys_mean", "S_vN system (bits)"),
    ("S_renyi_2_sys_mean", "S_Rényi-2 system = ancilla (bits)"),
    ("S_half_mean", "half-system S_vN (bits)"),
    ("S_vN_1q_mean", "mean single-qubit S_vN (bits)"),
    ("S_renyi_2_1q_mean", "mean single-qubit S_Rényi-2 (bits)"),
]
rows = load_sweep(sweep_dir)
METRICS = [m for m in METRICS if all(m[0] in r for r in rows)]
table = summarize(rows, by=("N", "p"), metrics=[key for key, _ in METRICS])
print(f"{len(rows)} runs")
print(f"{'N':>3} {'p':>5} {'runs':>4} " + " ".join(f"{key:>20}" for key, _ in METRICS))
for r in table:
    values = " ".join(
        f"{r[f'{key}_mean']:>9.4f} ± {r[f'{key}_sem']:<8.4f}" for key, _ in METRICS
    )
    print(f"{r['N']:>3} {r['p']:>5.2f} {r['n_runs']:>4} {values}")

## Metrics against p

Two figures, one row of panels each (issue #174): the performance of the readout, then the entanglement of the final state. Each curve is the mean over the seeds ± the standard error (`std / sqrt(n_runs)`); with a single seed the error bars are those each run reports, or none.

A panel can carry companion curves in the color of the same `N`, each with its own error bars:

- accuracy: the chance level (`accuracy_baseline`, majority sign), dashed. Only the gap to the chance level measures what the readout learns from `M`;
- ancilla and single-qubit entropies: the Rényi-2 entropy, dashed.

The gray dotted lines mark the bounds: 1 bit for the ancilla and single-qubit entropies.

Each figure and the data it shows are saved in the sweep folder, as `performance_vs_p.png` / `.json` and `entanglement_vs_p.png` / `.json` (`QELM.utils.plots.save_figure`).

In [ ]:
import matplotlib.pyplot as plt

from QELM.utils.plots import (
    color_map,
    draw_curve,
    save_figure,
    sizes,
    subplots,
    sweep_curve,
)

# One panel per entry: (key, title, y label, companions, bound). A companion
# is (key, what, line style), drawn in the color of the main curve of the
# same N; bound is a gray dotted level, or None.
PERFORMANCE = [
    (
        "accuracy",
        "Accuracy of label Y",
        "accuracy (test)",
        [
            ("accuracy_baseline", "chance (majority sign)", "--"),
        ],
        None,
    ),
    ("r2", "R2 of the prediction", "R2 (test)", [], None),
]
ENTANGLEMENT = [
    (
        "S_vN_mean",
        "Ancilla entropy",
        "S(R) (bits)",
        [("S_renyi_2_sys_mean", "Rényi-2", "--")],
        1.0,
    ),
    ("S_half_mean", "Half-system entropy", "S_half (bits)", [], None),
    (
        "S_vN_1q_mean",
        "Mean single-qubit entropy",
        "mean S(rho_k) (bits)",
        [("S_renyi_2_1q_mean", "Rényi-2", "--")],
        1.0,
    ),
]


def available(panels):
    """The panels whose metric the sweep has, each with its companions
    likewise filtered (an older sweep lacks the newer metrics)."""
    has = {key for key, _ in METRICS}
    return [
        (key, title, ylabel, [c for c in companions if c[0] in has], bound)
        for key, title, ylabel, companions, bound in panels
        if key in has
    ]


def draw_panels(axes, table, panels, N, color, label, linestyle="-"):
    """Draw the curves of size N on a row of panels: the metric with the
    given line style, and its companions with theirs."""
    for (key, _, _, companions, _), ax in zip(panels, axes, strict=True):
        draw_curve(
            ax, sweep_curve(table, N, "p", key), color, label=label, linestyle=linestyle
        )
        for other, what, style in companions:
            draw_curve(
                ax,
                sweep_curve(table, N, "p", other),
                color,
                label=f"{label}, {what}",
                linestyle=style,
            )


def finish_panels(axes, panels):
    """Bounds, labels, titles and legends of a row of panels."""
    for (_, title, ylabel, _, bound), ax in zip(panels, axes, strict=True):
        if bound is not None:
            ax.axhline(bound, color="gray", linestyle=":", label=f"{bound:g} bit")
        ax.set(xlabel="measurement probability p", ylabel=ylabel, title=title)
        ax.legend()


Ns = sizes(table)
colors = color_map([f"N{N}" for N in Ns])
figures = {}
axes_of = {}
for name, panels, title in (
    ("performance_vs_p", PERFORMANCE, "QELM on M: performance of the readout"),
    ("entanglement_vs_p", ENTANGLEMENT, "QELM on M: entanglement of the final state"),
):
    panels = available(panels)
    fig, axes = subplots(
        1, len(panels), figsize=(6.5 * len(panels), 5.5), squeeze=False
    )
    for N in Ns:
        draw_panels(axes[0], table, panels, N, colors[f"N{N}"], f"N = {N}")
    finish_panels(axes[0], panels)
    fig.suptitle(title)
    axes_of[name] = axes[0]
    figures[name] = save_figure(fig, sweep_dir, name, data={"table": table})
    print(f"figure: {figures[name]}")
    plt.show()

## Scaling with p for several fields h and sizes N

The same experiment, swept again over `p` in `[0, 1]` for two transverse fields and two sizes (issue #174). At `h = 10 J` (the config default) the large `Z` term nearly conserves the charge of each qubit, so each measurement reveals a lot about `Y` and the entanglement builds slowly between measurements. At `h = J` the `X_i X_j` couplings move the charge around: the state is more entangled, and the record must catch the charge as it spreads. Comparing `N = 4` and `N = 6` shows how the curves move with the size, the first sign of a transition, whose crossing point sharpens as `N` grows.

Three panels: the accuracy (the dotted gray line is chance, 0.5), the R2, and the half-system entropy `S_half`. One color per `N`, one line style per `h`: solid for `h = 10 J`, dashed for `h = J`.

The sweep goes to its own folder, `outputs/QELM_M_h/<date_time>/N<N>_h<h>_seed<seed>_p<p>/` (`H_VALUES` of the launcher), and uses the seeds and `L` of the parameters above. The figure is saved there as `scaling_vs_p.png` and `scaling_vs_p.json`.

In [ ]:
SCALING_N_VALUES = os.environ.get("SCALING_N_VALUES", "4 6")
H_VALUES = os.environ.get("H_VALUES", "1 10")  # TFIM.h, in units of TFIM.J
SCALING_P_VALUES = os.environ.get(
    "SCALING_P_VALUES", "0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0"
)
SCALING_EXPERIMENT = "QELM_M_h"

if RUN:
    scaling_dir = new_sweep_dir(OUT_ROOT, SCALING_EXPERIMENT)
    env = {
        **os.environ,
        "MODEL": "TFIM",
        "L": L,
        "N_VALUES": SCALING_N_VALUES,
        "H_VALUES": H_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": SCALING_P_VALUES,
        "OUT_DIR": str(scaling_dir),
    }
    n_runs = (
        len(SCALING_N_VALUES.split())
        * len(H_VALUES.split())
        * len(SEEDS.split())
        * len(SCALING_P_VALUES.split())
    )
    run_with_progress(
        ["bash", "experiments/QELM_M/exp_QELM_M.sh"], total=n_runs, env=env
    )
else:
    scaling_dir = latest_sweep(OUT_ROOT / SCALING_EXPERIMENT)
print(f"sweep: {scaling_dir}")

SCALING_METRICS = [
    ("accuracy", "Accuracy of label Y", "accuracy (test)"),
    ("r2", "R2 of the prediction", "R2 (test)"),
    ("S_half_mean", "Half-system entropy", "S_half (bits)"),
]
scaling_rows = load_sweep(
    scaling_dir,
    keys=(
        ("N", "N"),
        ("h", "TFIM.h"),
        ("seed", "seed"),
        ("p", "projective_measurement.p"),
    ),
)
scaling_table = summarize(
    scaling_rows, by=("N", "h", "p"), metrics=[key for key, _, _ in SCALING_METRICS]
)
print(f"{len(scaling_rows)} runs")
print(
    f"{'N':>3} {'h':>5} {'p':>5} {'runs':>4} "
    + " ".join(f"{key:>20}" for key, _, _ in SCALING_METRICS)
)
for r in scaling_table:
    values = " ".join(
        f"{r[f'{key}_mean']:>9.4f} ± {r[f'{key}_sem']:<8.4f}"
        for key, _, _ in SCALING_METRICS
    )
    print(f"{r['N']:>3} {r['h']:>5g} {r['p']:>5.2f} {r['n_runs']:>4} {values}")

In [ ]:
scaling_Ns = sizes(scaling_table)
hs = sorted({r["h"] for r in scaling_table}, reverse=True)
scaling_colors = color_map([f"N{N}" for N in scaling_Ns])
fig_s, axes_s = subplots(
    1, len(SCALING_METRICS), figsize=(6.5 * len(SCALING_METRICS), 5.5), squeeze=False
)
for N in scaling_Ns:
    for k, h in enumerate(hs):
        rows_h = [r for r in scaling_table if r["h"] == h]
        for (key, _, _), ax in zip(SCALING_METRICS, axes_s[0], strict=True):
            draw_curve(
                ax,
                sweep_curve(rows_h, N, "p", key),
                scaling_colors[f"N{N}"],
                label=f"N = {N}, h = {h:g} J",
                linestyle="-" if k == 0 else "--",
            )
for (key, title, ylabel), ax in zip(SCALING_METRICS, axes_s[0], strict=True):
    if key == "accuracy":
        ax.axhline(0.5, color="gray", linestyle=":", label="chance")
    ax.set(xlabel="measurement probability p", ylabel=ylabel, title=title)
    ax.legend()
fig_s.suptitle("QELM on M against p, for several fields h and sizes N")
scaling_figure = save_figure(
    fig_s, scaling_dir, "scaling_vs_p", data={"table": scaling_table}
)
print(f"figure: {scaling_figure}")
plt.show()